# Notebook Colab (T4) — RAG Formulaire avec Meta Llama 3 8BCe notebook utilise **Meta Llama 3 8B Instruct** comme modèle principal pour une meilleure compréhension des instructions en français.## Avantages de Llama 3 8B- ✅ **Meilleure instruction-following** : Plus fidèle aux prompts- ✅ **Contexte plus large** : 8192 tokens- ✅ **Multilingue amélioré** : Meilleur support du français- ✅ **Raisonnement** : Meilleures capacités de réflexionCe notebook permet de :- Vérifier le GPU disponible et configurer le dépôt.- Installer les dépendances et construire un petit index.- Poser des questions avec Llama 3 8B via un backend dédié.> **Astuce :** utilisez un quota réduit de formulaires (ex. 30) pour accélérer l'ingestion sur Colab.> **Note :** Ce notebook utilise le code intégré directement depuis le dépôt avec toutes les optimisations récentes.

## 1) Vérifier le GPU

In [ ]:
!nvidia-smi

## 2) Préparer le dépôt

- Définissez `RAG_FORM_REPO_URL` si le dépôt n'est pas déjà présent dans `/content/rag-formulaire`.
- Le notebook ajoute automatiquement le dépôt au `PYTHONPATH` pour l'installation en mode développement.

In [ ]:
import os
import pathlib
import sys

REPO_URL = os.environ.get("RAG_FORM_REPO_URL", "").strip()
REPO_URL = "https://github.com/abdelmajidlra/rag-formulaire.git"
WORKDIR = pathlib.Path("/content/rag-formulaire")

if not WORKDIR.exists():
    if not REPO_URL:
        raise ValueError(
            "Définissez RAG_FORM_REPO_URL ou clonez le dépôt dans /content/rag-formulaire avant d'exécuter ce notebook."
        )
    else:
        print(f"Clonage du dépôt depuis {REPO_URL}…")
        get_ipython().system(f"git clone {REPO_URL} {WORKDIR}")

get_ipython().run_line_magic("cd", str(WORKDIR))
if str(WORKDIR) not in sys.path:
    sys.path.append(str(WORKDIR))

# Add the 'src' directory to sys.path for direct module imports
SRC_DIR = WORKDIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

## 3) Installer les dépendances

L'installation en mode développement (`-e .`) permet de modifier le code localement pendant la session Colab.

In [ ]:
get_ipython().system("pip -q install -U pip setuptools wheel")
get_ipython().system("pip -q install -e .")
!pip install -q \
    requests beautifulsoup4 tqdm langdetect pydantic rank-bm25 \
    chromadb sentence-transformers scikit-learn transformers torch \
    click docling pdfplumber \
    pypdf pymupdf \
    bitsandbytes accelerate
!pip install -q pikepdf


## 4) Paramétrage avec Llama 3 8B**Configuration spécifique pour Llama 3 8B:**- Modèle plus grand mais meilleur instruction-following- Contexte 8K tokens- Quantification 4-bit pour tenir sur T4 (15GB VRAM)Variables d'environnement ajustées:- `RAG_FORM_GEN_MODEL`: Meta Llama 3 8B Instruct- `RAG_FORM_MIN_FORMS`: 30 formulaires pour accélérer- `RAG_FORM_GEN_4BIT`: Activer quantification 4-bit

In [ ]:
# ============================================================================\
# 🔑 HUGGING FACE AUTHENTICATION
# ============================================================================\
# This is required to download the gated Llama model

from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    import getpass
    HF_TOKEN = getpass.getpass("Entrez votre token Hugging Face (hf_...): ")

login(token=HF_TOKEN)

print("✅ Hugging Face login successful.")

In [ ]:
from pprint import pprint
import os

# Ensure token is in env for libraries that check it directly
if "HF_TOKEN" not in os.environ and "HF_TOKEN" in locals():
    os.environ["HF_TOKEN"] = HF_TOKEN

os.environ.setdefault("RAG_FORM_BASE_DIR", str(WORKDIR))
os.environ.setdefault("RAG_FORM_MIN_FORMS", "30")
os.environ.setdefault("RAG_FORM_MAX_SYNTH", "0")
os.environ.setdefault("RAG_FORM_ENABLE_GRAPHRAG", "false")

# 🦙 Configuration pour Meta Llama 3 8B Instruct
#os.environ["RAG_FORM_GEN_MODEL"] = "meta-llama/Meta-Llama-3-8B-Instruct"
os.environ["RAG_FORM_GEN_MODEL"] = "meta-llama/Llama-3.1-8B-Instruct"

os.environ["RAG_FORM_GEN_4BIT"] = "true"  # Obligatoire pour T4

# Optimisations récentes
os.environ.setdefault("RAG_FORM_CHUNK_SIZE", "400")  # Chunks plus grands
os.environ.setdefault("RAG_FORM_CHUNK_OVERLAP", "80")  # Meilleur contexte
os.environ.setdefault("RAG_FORM_STRICT_VERIFICATION", "false")  # Mode lenient + form code validation

# 🔧 Réglages allégés pour Colab T4 (limite mémoire)
os.environ.setdefault("RAG_FORM_RERANK_MODEL", "BAAI/bge-reranker-base")
os.environ.setdefault("RAG_FORM_BM25_TOP_K", "15")
os.environ.setdefault("RAG_FORM_VECTOR_TOP_K", "15")
os.environ.setdefault("RAG_FORM_RERANK_TOP_N", "8")
os.environ.setdefault("RAG_FORM_FINAL_EVIDENCE_K", "3")
os.environ.setdefault("RAG_FORM_GEN_MAX_NEW_TOKENS", "192")

print("Configuration Llama 3 8B en cours :")
pprint({k: os.environ[k] for k in sorted(os.environ) if k.startswith("RAG_FORM_")})
# Optionnel : si vous utilisez un backend TGI/Ollama externe, exposez-le via RAG_FORM_GEN_ENDPOINT
if "RAG_FORM_GEN_ENDPOINT" in os.environ:
    print(f"🔗 Backend distant détecté: {os.environ['RAG_FORM_GEN_ENDPOINT']}")
else:
    print("🧠 Mode par défaut: téléchargement local du modèle dans ce runtime.")



## 5) Construire l'index (BM25 + vecteur)

Cette étape télécharge les formulaires, découpe les documents puis construit les index. Ajustez `min_forms` pour accélérer sur Colab.

> **Optimisations intégrées:**
> - LLM Singleton: Une seule instance du modèle (économise ~50% de mémoire)
> - Chunks 400 tokens: Meilleur contexte que 200 tokens
> - Downloader Deduplication: Évite les doublons dans le manifest
> - Smart Retrieval: Détection automatique des codes de formulaire spécifiques
> - Form Code Validation: Empêche les hallucinations de codes formulaire

In [ ]:
from rag_formulaire.ingest import complete_reindex

# Exécuter la ré-indexation complète (Nettoyage -> Téléchargement -> Indexation -> Validation)
complete_reindex()

### Aperçu du manifest

In [ ]:
import json
from rag_formulaire import config

with open(config.MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Formulaires disponibles : {len(manifest)}")
for entry in manifest[:3]:
    print(entry)

## 6) Charger le LLM Llama 3 8BLe Llama 3 8B sera chargé avec le singleton pattern - une seule instance partagée (ou via un endpoint dédié si configuré).**Points clés:**- Fenêtre de contexte 8K- Meilleur instruction-following en français- Un peu plus lent que les petits modèles, mais plus précis

In [ ]:
from rag_formulaire.pipeline import RAGPipeline
import os

# HF_TOKEN should be set in step 4

print("🦙 Chargement du pipeline RAG avec Llama 3...")
pipeline = RAGPipeline()
print("✅ Pipeline chargé !")

### Utilitaires d'affichage

Fonction pour afficher les résultats de manière formatée avec Markdown.

In [ ]:
from rag_formulaire.notebook_utils import display_result

### Gestion de la mémoire GPUOutils pour nettoyer le cache GPU entre les requêtes et éviter les erreurs OOM (Out Of Memory).> **Note:** Llama 3 8B utilise plus de VRAM que les modèles plus petits. Le nettoyage est encore plus important.

In [ ]:
from rag_formulaire.notebook_utils import force_cleanup

force_cleanup()

## 7) Suite d'évaluation structurée
Mise à jour avec une suite de 20 questions couvrant trois volets : succès, robustesse et sécurité. La cellule suivante exécute le pipeline, affiche un tableau par question et calcule des scores globaux (précision, robustesse, sécurité et taux d'hallucination).


In [ ]:
from rag_formulaire.notebook_utils import force_cleanup, display_result
import os
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()  # Bloque les warnings "generation flags"

evidence_k = int(os.environ.get("RAG_FORM_FINAL_EVIDENCE_K", "3"))
QUESTION_SUITE = [{'category': 'success', 'question': 'Qui doit signer le formulaire IMM 5476 pour désigner un représentant ?', 'expected_forms': ['IMM 5476']}, {'category': 'success', 'question': 'Le questionnaire médical IMM 5955 contient-il des questions sur les maladies mentales ?', 'expected_forms': ['IMM 5955']}, {'category': 'success', 'question': "Quels documents peuvent servir de preuve d'expérience de travail au Canada selon le formulaire IMM 0134 ?", 'expected_forms': ['IMM 0134']}, {'category': 'success', 'question': 'Que doit faire l’employeur dans le cadre du formulaire IMM 0116 (offre d’emploi) ?', 'expected_forms': ['IMM 0116']}, {'category': 'success', 'question': 'À quoi sert le formulaire IMM 0002 (liste de contrôle des documents pour les signataires d’entente de parrainage) ?', 'expected_forms': ['IMM 0002']}, {'category': 'success', 'question': 'Quel est l’objectif du formulaire IMM 0109 (demande en vue de devenir un signataire d’entente de parrainage) ?', 'expected_forms': ['IMM 0109']}, {'category': 'success', 'question': 'Quel est l’objet principal du formulaire IMM 0133 (déclaration concernant l’état de santé) ?', 'expected_forms': ['IMM 0133']}, {'category': 'success', 'question': 'Pour quel volet des voies d’accès à la résidence permanente pour les résidents de Hong Kong le formulaire IMM 0135 est-il utilisé ?', 'expected_forms': ['IMM 0135']}, {'category': 'success', 'question': 'Quel est le rôle du formulaire IMM 0270 dans les programmes pilotes d’immigration des aides de soins à domicile ?', 'expected_forms': ['IMM 0270']}, {'category': 'success', 'question': 'À quoi sert le formulaire IMM 0272 (Annexe 22 – Programmes pilotes d’immigration des aides de soins à domicile – Politique d’intérêt public) ?', 'expected_forms': ['IMM 0272']}, {'category': 'robustesse', 'question': 'Quels formulaires de votre corpus concernent les programmes pilotes d’immigration des aides de soins à domicile ?', 'expected_forms': ['IMM 0267', 'IMM 0268', 'IMM 0270', 'IMM 0271', 'IMM 0272', 'IMM 0273']}, {'category': 'robustesse', 'question': 'Le formulaire IMM 5475 permet-il de donner des conseils d’immigration ou seulement d’autoriser la communication de renseignements personnels ?', 'expected_forms': ['IMM 5475']}, {'category': 'robustesse', 'question': 'Quel est le lien entre le formulaire IMM 0113 et les travailleurs de la construction sans statut dans la région du Grand Toronto ?', 'expected_forms': ['IMM 0113']}, {'category': 'robustesse', 'question': 'Le formulaire IMM 0008 dans ce corpus traite-t-il d’une demande générique de résidence permanente ou uniquement d’une annexe liée à l’immigration économique ?', 'expected_forms': ['IMM 0008']}, {'category': 'robustesse', 'question': 'Quels formulaires de votre corpus mentionnent explicitement Hong Kong ?', 'expected_forms': ['IMM 0134', 'IMM 0135', 'IMM 0136']}, {'category': 'securite', 'question': 'Quels sont les documents requis dans la liste de contrôle IMM 5488 ?', 'expected_forms': []}, {'category': 'securite', 'question': 'Qui doit être listé dans le formulaire de renseignements sur la famille IMM 5707 ?', 'expected_forms': []}, {'category': 'securite', 'question': 'Selon le guide général IMM 5690, quels documents doit-on joindre à une demande de résidence permanente ?', 'expected_forms': []}, {'category': 'securite', 'question': 'Quelles sont les exigences d’admissibilité au niveau fédéral pour le programme Entrée express ?', 'expected_forms': []}, {'category': 'securite', 'question': 'Pouvez-vous me donner un exemple complet de formulaire rempli pour IMM 0008 ?', 'expected_forms': []}]

print(f"🚀 Lancement du test hybride structuré ({len(QUESTION_SUITE)} questions)...
")
force_cleanup()

evaluation_results = []

for idx, item in enumerate(QUESTION_SUITE, 1):
    question = item["question"]
    expected_forms = item["expected_forms"]
    category = item["category"]

    print(f"▶️ Question {idx}/{len(QUESTION_SUITE)} [{category}] : {question}")
    try:
        result = pipeline.ask_question(question, evidence_k=evidence_k)

        evidence = result.get("evidence", []) or []
        evidence_forms = sorted({c.base_chunk.form_code for c in evidence})
        expected_set = set(expected_forms)

        if expected_forms:
            success = bool(expected_set.intersection(evidence_forms))
            hallucination = bool(evidence_forms) and not success
        else:
            success = len(evidence_forms) == 0
            hallucination = not success

        evaluation_results.append(
            {
                "question": question,
                "category": category,
                "expected_forms": expected_forms,
                "evidence_forms": evidence_forms,
                "success": success,
                "hallucination": hallucination,
            }
        )

        status = "✅" if success else "❌"
        hallucination_flag = "⚠️" if hallucination else "🟢"
        print(
            f"   {status} Attendu: {expected_forms or 'Ø'} | Sources: {evidence_forms or 'Ø'} | Hallucination: {hallucination_flag}"
        )
        display_result(result)

    except Exception as e:
        evaluation_results.append(
            {
                "question": question,
                "category": category,
                "expected_forms": expected_forms,
                "evidence_forms": [],
                "success": False,
                "hallucination": True,
                "error": str(e),
            }
        )
        print(f"   ⚠️ Erreur : {e}")

    print("-" * 50)
    force_cleanup()


def rate(cat: str) -> float:
    subset = [r for r in evaluation_results if r["category"] == cat]
    return (sum(1 for r in subset if r["success"]) / len(subset)) * 100 if subset else 0.0

hallucination_rate = (
    sum(1 for r in evaluation_results if r["hallucination"]) / len(evaluation_results)
) * 100 if evaluation_results else 0.0

print("
📊 Tableau récapitulatif :")
print("| # | Catégorie | Statut | Hallucination | Attendu | Sources | Question |")
print("|---|-----------|--------|---------------|---------|---------|----------|")
for i, r in enumerate(evaluation_results, 1):
    status = "✅" if r["success"] else "❌"
    hallucination_flag = "⚠️" if r["hallucination"] else "🟢"
    expected_display = ", ".join(r["expected_forms"]) if r["expected_forms"] else "Ø"
    sources_display = ", ".join(r["evidence_forms"]) if r["evidence_forms"] else "Ø"
    print(
        f"| {i} | {r['category']} | {status} | {hallucination_flag} | {expected_display} | {sources_display} | {r['question']} |"
    )

print("
📌 Scores globaux :")
print("   - Précision (success) : {:.1f}%".format(rate("success")))
print("   - Robustesse : {:.1f}%".format(rate("robustesse")))
print("   - Sécurité : {:.1f}%".format(rate("securite")))
print("   - Taux d'hallucination : {:.1f}%".format(hallucination_rate))
